# Phase 2 — Pandas From True Basics (Addis Ababa Edition)

You've already used Pandas once (the earlier notebook). This one is different: it builds the *concepts* underneath those methods, so `groupby()` and `.loc[]` stop being "commands that work" and become things you actually understand.

**Sections:**
1. What a Series and DataFrame actually are
2. The Index — the part everyone skips and later gets confused by
3. Reading real files (`read_csv`) and what can go wrong
4. Selection: `.loc` vs `.iloc`, and why they're different
5. Boolean masking — how filtering actually works under the hood
6. GroupBy — what's really happening in three steps (split-apply-combine)
7. Merging — the four join types, visually
8. Missing data — `NaN`, and why it's not just "empty"
9. `apply()` and vectorization — why loops are usually the wrong tool
10. Basic plotting
11. Checkpoint — rebuild your earlier notebook's mini-challenge from scratch, no notes


## 1. What a Series and DataFrame actually are

A **Series** is not "a column" in the abstract sense — it's two aligned arrays glued together: an **index** (labels) and **values** (the data). Every value has a label, always.

A **DataFrame** is a dict of Series that all share the *same* index. That's it — that's the whole structure. Once this clicks, a lot of Pandas behavior (like why `df["col"] + df["other_col"]` aligns automatically) stops being mysterious.

In [ ]:
import pandas as pd
import numpy as np

# A Series is index + values, always
populations = pd.Series(
    [328900, 401000, 235100],
    index=["Bole", "Yeka", "Kirkos"]
)
print(populations)
print()
print("Index:", populations.index.tolist())
print("Values:", populations.values)

In [ ]:
# A DataFrame is multiple Series sharing one index
subcity_df = pd.DataFrame({
    "population": [328900, 401000, 235100],
    "area_km2": [122.8, 85.9, 14.6]
}, index=["Bole", "Yeka", "Kirkos"])

print(subcity_df)
print()
# Proof: pulling one column out gives you back a Series with the SAME index
print(type(subcity_df["population"]))
print(subcity_df["population"].index.tolist())

### Exercise 1.1
Create a Series called `income` with values `[18500, 12200, 15800]` and index `["Bole", "Yeka", "Kirkos"]`. Then create a second Series `population` with `[328900, 401000, 235100]` and the *same* index.

Add them together how you'd expect: `income + population` doesn't make sense conceptually, so instead compute `population * 2` and print it — notice the index labels carry through automatically. This is the "alignment" behavior that makes Pandas powerful (and sometimes confusing if your indexes don't match).

In [ ]:
# Your code here


<details>
<summary>Solution 1.1</summary>

```python
income = pd.Series([18500, 12200, 15800], index=["Bole", "Yeka", "Kirkos"])
population = pd.Series([328900, 401000, 235100], index=["Bole", "Yeka", "Kirkos"])

doubled = population * 2
print(doubled)
```

Notice the labels (`Bole`, `Yeka`, `Kirkos`) stayed attached the whole time — that's the index doing its job.
</details>

## 2. The Index — the part everyone skips

By default, when you build a DataFrame from a list of dicts or a CSV, Pandas assigns a boring `0, 1, 2, 3...` index. But you can set any column as the index — and this matters a lot once you start merging/joining data, because joins happen *on the index or a specified key column*.

Key methods:
- `.set_index("col")` — make a column the index
- `.reset_index()` — turn the index back into a regular column, replace it with `0,1,2...`
- `df.loc["label"]` — select by index label (only works cleanly with a meaningful index)

In [ ]:
subcity_df2 = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos"],
    "population": [328900, 401000, 235100]
})
print("Default index:")
print(subcity_df2)
print()

subcity_df2 = subcity_df2.set_index("sub_city")
print("After set_index:")
print(subcity_df2)
print()

# Now .loc works with sub-city names directly
print(subcity_df2.loc["Yeka"])

### Exercise 2.1
Build a DataFrame with columns `sub_city` and `avg_income_birr` for at least 4 sub-cities. Set `sub_city` as the index. Then use `.loc[]` to print the income of one specific sub-city by name.

In [ ]:
# Your code here


<details>
<summary>Solution 2.1</summary>

```python
income_df = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos", "Lideta"],
    "avg_income_birr": [18500, 12200, 15800, 11900]
})
income_df = income_df.set_index("sub_city")
print(income_df.loc["Kirkos"])
```
</details>

## 3. Reading real files — `read_csv` and what can go wrong

`pd.read_csv()` does a lot for you automatically (type inference, header detection) — which is exactly why it silently gets things wrong sometimes. Common real-world snags:

- Wrong delimiter (some exports use `;` or tab instead of `,`) → fix with `sep=";"`
- Numbers read as text because of stray characters (e.g. `"1,200"` with a comma) → often needs cleaning after reading
- Extra blank rows at the top → `skiprows=n`
- Encoding issues (rare with English text, common with Amharic text) → try `encoding="utf-8"` explicitly

Let's create a deliberately slightly-messy CSV and read it.

In [ ]:
# Create a sample CSV with a messy quirk: population has a thousands-comma
csv_content = """sub_city,population,area_km2
Bole,"328,900",122.8
Yeka,"401,000",85.9
Kirkos,"235,100",14.6
"""

with open("messy_subcities.csv", "w") as f:
    f.write(csv_content)

# Naive read
df_naive = pd.read_csv("messy_subcities.csv")
print(df_naive.dtypes)
print(df_naive)

Notice: `population` read in as `object` (Pandas' term for text/mixed data), not a number — because of the comma inside the quotes. This is exactly the kind of thing that breaks a `.sum()` or `.mean()` call later with a confusing error, unless you catch it here.

### Exercise 3.1
Fix `df_naive`'s `population` column: remove the commas and convert it to an integer type. 

Tip: `.str.replace(",", "")` works on a Series of strings, then wrap with `.astype(int)`.

In [ ]:
# Your code here


<details>
<summary>Solution 3.1</summary>

```python
df_naive["population"] = df_naive["population"].str.replace(",", "").astype(int)
print(df_naive.dtypes)
print(df_naive)
```
</details>

## 4. Selection: `.loc` vs `.iloc`

This trips up almost everyone at first. The difference:

- **`.loc[]`** — selects by **label** (the index value, or column name)
- **`.iloc[]`** — selects by **integer position** (0, 1, 2... regardless of what the labels actually are)

If your index is the default `0,1,2...`, they often look like they do the same thing — which is exactly why people get confused later when the index is something else (like sub-city names) and `.iloc[0]` vs `.loc[0]` suddenly behave completely differently.

In [ ]:
demo_df = pd.DataFrame({
    "population": [328900, 401000, 235100]
}, index=["Bole", "Yeka", "Kirkos"])

print(demo_df)
print()
print("loc['Yeka']:")
print(demo_df.loc["Yeka"])   # by label
print()
print("iloc[1]:")
print(demo_df.iloc[1])       # by position — happens to be the same row here

### Exercise 4.1
Using `demo_df` above, select the **first two rows** using `.iloc`, then select **only "Bole" and "Kirkos"** using `.loc` (pass a list of labels).

In [ ]:
# Your code here


<details>
<summary>Solution 4.1</summary>

```python
print(demo_df.iloc[0:2])
print()
print(demo_df.loc[["Bole", "Kirkos"]])
```
</details>

## 5. Boolean masking — how filtering actually works

`df[df["population"] > 300000]` looks like magic. It isn't — `df["population"] > 300000` alone produces a Series of `True`/`False` values (one per row). Passing that Series into `df[...]` just keeps the rows where it's `True`. Let's see both steps separately.

In [ ]:
subcity_df3 = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos", "Lideta"],
    "population": [328900, 401000, 235100, 210300]
})

# Step 1: the mask itself
mask = subcity_df3["population"] > 300000
print("The mask (just True/False):")
print(mask)
print()

# Step 2: using the mask to filter
print("Filtered result:")
print(subcity_df3[mask])

### Exercise 5.1
Build the mask for "population is between 200000 and 350000 (inclusive)" as its own variable first (print it), then apply it to filter `subcity_df3`.

Tip: combine two conditions with `&`, and wrap each condition in parentheses: `(cond1) & (cond2)`.

In [ ]:
# Your code here


<details>
<summary>Solution 5.1</summary>

```python
mask = (subcity_df3["population"] >= 200000) & (subcity_df3["population"] <= 350000)
print(mask)
print()
print(subcity_df3[mask])
```
</details>

## 6. GroupBy — split, apply, combine

`groupby()` isn't one operation — it's three, always in this order:

1. **Split** — break the DataFrame into groups based on a column's values
2. **Apply** — run some function (sum, mean, count...) on each group separately
3. **Combine** — stitch the per-group results back into one output

Let's see the "split" step in isolation before jumping to the shortcut syntax.

In [ ]:
transit_data = pd.DataFrame({
    "sub_city": ["Bole", "Bole", "Yeka", "Yeka", "Kirkos"],
    "mode": ["Bus", "Rail", "Bus", "Taxi", "Bus"],
    "boardings": [1200, 800, 950, 400, 600]
})

grouped = transit_data.groupby("sub_city")

# The "split" step created groups — you can look at one directly
print("Just the Bole group:")
print(grouped.get_group("Bole"))
print()

# The "apply + combine" happens when you call an aggregation
print("Sum per sub-city (apply + combine):")
print(grouped["boardings"].sum())

### Exercise 6.1
Group `transit_data` by `mode` instead of `sub_city`. First print just the group for `"Bus"` using `.get_group()`, then print the mean `boardings` per mode.

In [ ]:
# Your code here


<details>
<summary>Solution 6.1</summary>

```python
grouped_by_mode = transit_data.groupby("mode")
print(grouped_by_mode.get_group("Bus"))
print()
print(grouped_by_mode["boardings"].mean())
```
</details>

## 7. Merging — the four join types

Merging combines two DataFrames based on a shared key column. The `how` parameter decides what happens to rows that *don't* have a match on the other side:

- **`inner`** — keep only rows where the key exists in **both** DataFrames
- **`left`** — keep **all** rows from the left DataFrame, fill with `NaN` if no match on the right
- **`right`** — same, but keep all rows from the right
- **`outer`** — keep **all** rows from **both**, `NaN` wherever there's no match

This matters a lot for your work: if you merge transit stops with sub-city data and a sub-city has zero stops, an `inner` join would silently *drop that sub-city entirely* — which is exactly the kind of bug that skews an "underserved areas" analysis without you noticing.

In [ ]:
subcities = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos", "Akaky Kaliti"],
    "population": [328900, 401000, 235100, 195400]
})

# Note: no stop data at all for "Akaky Kaliti", and stop data exists for a city not in subcities
stop_counts = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Nifas Silk-Lafto"],
    "stop_count": [12, 9, 15]
})

print("INNER join — only cities present in BOTH:")
print(pd.merge(subcities, stop_counts, on="sub_city", how="inner"))
print()

print("LEFT join — ALL sub-cities kept, even Akaky Kaliti with no stops:")
print(pd.merge(subcities, stop_counts, on="sub_city", how="left"))
print()

print("OUTER join — everything from both sides:")
print(pd.merge(subcities, stop_counts, on="sub_city", how="outer"))

### Exercise 7.1
Using the same `subcities` and `stop_counts` DataFrames above: do a `left` merge, then explain in a comment why `left` (not `inner`) is the correct choice for an "underserved areas" analysis — i.e., what would go wrong with `inner` here.

In [ ]:
# Your code here (merge + a comment explaining the reasoning)


<details>
<summary>Solution 7.1</summary>

```python
result = pd.merge(subcities, stop_counts, on="sub_city", how="left")
print(result)

# Reasoning: an inner join would silently DROP "Akaky Kaliti" because it has
# no matching stop_count row. But "no transit stops" is exactly the finding
# we care about for an underserved-areas analysis — dropping it would hide
# the very sub-city we're trying to flag. Left join keeps it with NaN,
# which we'd then fillna(0) to correctly show zero stops.
```
</details>

## 8. Missing data — `NaN` isn't just "empty"

`NaN` (Not a Number) is a special float value that propagates: almost any operation involving `NaN` returns `NaN`, including comparisons — `NaN == NaN` is actually `False`. This is why you check for missing values with `.isna()`, never `== NaN`.

In [ ]:
test_series = pd.Series([1, 2, np.nan, 4])

print(test_series == np.nan)   # all False, even the actual NaN — this is the trap
print()
print(test_series.isna())      # correct way to check
print()
print(test_series.sum())       # NaN is skipped automatically by sum()
print(test_series.mean())      # also skipped automatically

### Exercise 8.1
Given the Series below, replace missing values with the column's mean (not 0 — using the actual mean is usually more representative for something like boardings data).

```python
boardings = pd.Series([1200, np.nan, 950, 400, np.nan, 600])
```

In [ ]:
boardings = pd.Series([1200, np.nan, 950, 400, np.nan, 600])

# Your code here


<details>
<summary>Solution 8.1</summary>

```python
boardings_filled = boardings.fillna(boardings.mean())
print(boardings_filled)
```
</details>

## 9. `apply()` and vectorization — why loops are usually wrong

You *can* loop over a DataFrame row by row, but Pandas is built on NumPy arrays under the hood — operating on a whole column at once ("vectorized") is dramatically faster than looping, and is idiomatic Pandas. `apply()` is the middle ground for logic that doesn't have a built-in vectorized function.

In [ ]:
import time

big_series = pd.Series(range(200000))

# Slow way: manual loop
start = time.time()
result_loop = []
for v in big_series:
    result_loop.append(v * 2)
loop_time = time.time() - start

# Fast way: vectorized
start = time.time()
result_vectorized = big_series * 2
vec_time = time.time() - start

print(f"Loop took: {loop_time:.4f}s")
print(f"Vectorized took: {vec_time:.4f}s")
print(f"Vectorized was ~{loop_time / vec_time:.0f}x faster")

### Exercise 9.1
Given `subcity_df3` (from Section 5), use `.apply()` with a small function to create a new column `income_tier` — but this time, instead of a simple threshold, classify based on **two columns at once**: if `population > 300000` **and** you'd guess it's a busier area, label `"Priority"`, otherwise `"Standard"`.

Tip: when using two columns in `apply()`, you need `axis=1` and your function receives the whole row.

In [ ]:
# Your code here


<details>
<summary>Solution 9.1</summary>

```python
def classify_row(row):
    if row["population"] > 300000:
        return "Priority"
    else:
        return "Standard"

subcity_df3["priority_tier"] = subcity_df3.apply(classify_row, axis=1)
print(subcity_df3)
```

Note: this particular example only uses one column, so a plain `subcity_df3["population"].apply(...)` (axis=0, single-column) would also work and be simpler. The `axis=1` pattern above matters once your classification genuinely needs *multiple* columns together, e.g. population AND area.
</details>

## 10. Basic plotting

Pandas has `.plot()` built in (wraps Matplotlib) — good for a quick look before you build a polished chart elsewhere.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

subcity_df3.set_index("sub_city")["population"].plot(kind="bar", title="Population by Sub-City")
plt.ylabel("Population")
plt.show()

### Exercise 10.1
Plot `avg_income_birr` (reuse `income_df` from Exercise 2.1, or rebuild it) as a horizontal bar chart (`kind="barh"`).

In [ ]:
# Your code here


<details>
<summary>Solution 10.1</summary>

```python
income_df2 = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos", "Lideta"],
    "avg_income_birr": [18500, 12200, 15800, 11900]
}).set_index("sub_city")

income_df2["avg_income_birr"].plot(kind="barh", title="Average Income by Sub-City")
plt.xlabel("Average Income (Birr)")
plt.show()
```
</details>

## 11. Checkpoint — rebuild the mini-challenge from scratch

Back in your first Pandas notebook, the final mini-challenge computed a "transit access score" per sub-city. Rebuild that entire pipeline here — **without opening the old notebook** — using everything from this notebook:

1. Create two DataFrames: `subcity_df` (sub_city, population) for at least 6 sub-cities, and `stops_df` (sub_city, stop_count) — deliberately leave out stop data for at least one sub-city, and include a sub-city in `stops_df` that isn't in `subcity_df`
2. Merge them with the **correct** join type (explain why in a comment)
3. Handle the resulting missing values correctly (`fillna`, not dropping rows)
4. Compute `transit_access_score = stop_count / (population / 10000)`
5. Sort and print the 3 lowest-scoring (most underserved) sub-cities

If you can do this cold, Phase 2 is genuinely done.

In [ ]:
# Your checkpoint code here


<details>
<summary>Reference solution</summary>

```python
subcity_df_final = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos", "Lideta", "Addis Ketema", "Akaky Kaliti"],
    "population": [328900, 401000, 235100, 210300, 345600, 195400]
})

stops_df_final = pd.DataFrame({
    "sub_city": ["Bole", "Yeka", "Kirkos", "Lideta", "Gulele"],  # Gulele isn't in subcity_df; Akaky Kaliti has no stops
    "stop_count": [12, 9, 6, 5, 8]
})

# LEFT join: keep every sub-city we actually track population for,
# even if it has zero stops (that's the point of the analysis).
# An inner join would silently drop Akaky Kaliti.
merged = pd.merge(subcity_df_final, stops_df_final, on="sub_city", how="left")
merged["stop_count"] = merged["stop_count"].fillna(0)

merged["transit_access_score"] = merged["stop_count"] / (merged["population"] / 10000)

underserved = merged.sort_values("transit_access_score").head(3)
print(underserved)
```
</details>

## Next steps

With Phase 2 done cold, you're ready for **Phase 3 — Working with Real, Messy Data** (encoding issues, regex cleaning, JSON handling) — which is a shorter phase and leads directly into re-approaching your `addis-transit-access` project's raw data with fresh eyes.
